In [1]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import mlflow.xgboost
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, mean_absolute_percentage_error
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
import lightgbm as lgb
from prophet import Prophet
import warnings
warnings.filterwarnings('ignore')

# Setup MLflow
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("NeuralRetail_AMX_DS_2026")

print("✅ MLflow setup complete!")
print(f"📊 Tracking URI: sqlite:///mlflow.db")

2026/06/12 12:06:48 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/12 12:06:48 INFO mlflow.store.db.utils: Updating database tables
2026/06/12 12:06:52 INFO mlflow.tracking.fluent: Experiment with name 'NeuralRetail_AMX_DS_2026' does not exist. Creating a new experiment.


✅ MLflow setup complete!
📊 Tracking URI: sqlite:///mlflow.db


In [2]:
df = pd.read_excel(r"C:\Users\Atharva More\Downloads\online_retail_II.xlsx",
                   sheet_name='Year 2010-2011')
df_clean = df.dropna(subset=['Customer ID'])
df_clean = df_clean[df_clean['Quantity'] > 0]
df_clean = df_clean[df_clean['Price'] > 0]
df_clean['TotalAmount'] = df_clean['Quantity'] * df_clean['Price']
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])
df_clean['Customer ID'] = df_clean['Customer ID'].astype(int)

snapshot_date = df_clean['InvoiceDate'].max() + pd.Timedelta(days=1)
rfm = df_clean.groupby('Customer ID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'Invoice': 'nunique',
    'TotalAmount': 'sum'
}).reset_index()
rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']
rfm['AvgOrderValue'] = rfm['Monetary'] / rfm['Frequency']
rfm['RecencyScore'] = pd.qcut(rfm['Recency'], 4, labels=[4,3,2,1]).astype(int)
rfm['FrequencyScore'] = pd.qcut(rfm['Frequency'].rank(method='first'), 4, labels=[1,2,3,4]).astype(int)
rfm['MonetaryScore'] = pd.qcut(rfm['Monetary'].rank(method='first'), 4, labels=[1,2,3,4]).astype(int)
rfm['RFMScore'] = rfm['RecencyScore'] + rfm['FrequencyScore'] + rfm['MonetaryScore']
rfm['Churn'] = (rfm['Recency'] > 90).astype(int)

features = ['Recency', 'Frequency', 'Monetary', 'AvgOrderValue',
            'RecencyScore', 'FrequencyScore', 'MonetaryScore', 'RFMScore']
X = rfm[features]
y = rfm['Churn']
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=features)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y)

print("✅ Data prepared!")

✅ Data prepared!


In [3]:
with mlflow.start_run(run_name="LogisticRegression_Baseline"):
    lr = LogisticRegression(random_state=42)
    lr.fit(X_train, y_train)
    lr_auc = roc_auc_score(y_test, lr.predict_proba(X_test)[:,1])
    
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("test_size", 0.2)
    mlflow.log_metric("auc_roc", lr_auc)
    mlflow.sklearn.log_model(lr, "logistic_regression_model")
    
    print(f"✅ Logistic Regression logged! AUC: {lr_auc:.4f}")

2026/06/12 12:11:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/12 12:11:31 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


✅ Logistic Regression logged! AUC: 1.0000


In [4]:
with mlflow.start_run(run_name="XGBoost_Churn"):
    xgb_model = xgb.XGBClassifier(
        n_estimators=200, max_depth=6,
        learning_rate=0.05, random_state=42, verbosity=0)
    xgb_model.fit(X_train, y_train)
    xgb_auc = roc_auc_score(y_test, xgb_model.predict_proba(X_test)[:,1])
    
    mlflow.log_param("model_type", "XGBoost")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("learning_rate", 0.05)
    mlflow.log_metric("auc_roc", xgb_auc)
    mlflow.xgboost.log_model(xgb_model, "xgboost_churn_model")
    
    print(f"✅ XGBoost logged! AUC: {xgb_auc:.4f}")

2026/06/12 12:12:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


✅ XGBoost logged! AUC: 1.0000


In [5]:
with mlflow.start_run(run_name="LightGBM_Churn"):
    lgb_model = lgb.LGBMClassifier(
        n_estimators=200, max_depth=6,
        learning_rate=0.05, random_state=42, verbosity=-1)
    lgb_model.fit(X_train, y_train)
    lgb_auc = roc_auc_score(y_test, lgb_model.predict_proba(X_test)[:,1])
    
    mlflow.log_param("model_type", "LightGBM")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 6)
    mlflow.log_metric("auc_roc", lgb_auc)
    mlflow.sklearn.log_model(lgb_model, "lightgbm_churn_model")
    
    print(f"✅ LightGBM logged! AUC: {lgb_auc:.4f}")

  File "C:\Users\Atharva More\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\Atharva More\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Atharva More\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\Atharva More\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
2026/06/12 12:13:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/12 12:13:03 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Pyth

✅ LightGBM logged! AUC: 1.0000


In [6]:
print("=" * 55)
print("📊 MLFLOW EXPERIMENT SUMMARY")
print("=" * 55)
print(f"✅ Logistic Regression AUC: {lr_auc:.4f}")
print(f"✅ XGBoost AUC:             {xgb_auc:.4f}")
print(f"✅ LightGBM AUC:            {lgb_auc:.4f}")
print()
print("💡 To view MLflow UI run in Anaconda Prompt:")
print("   cd 'C:\\Users\\Atharva More\\NeuralRetail'")
print("   mlflow ui")
print("   Then open: http://localhost:5000")
print("=" * 55)
print("✅ Day 15-20 MLflow Tracking Complete!")

📊 MLFLOW EXPERIMENT SUMMARY
✅ Logistic Regression AUC: 1.0000
✅ XGBoost AUC:             1.0000
✅ LightGBM AUC:            1.0000

💡 To view MLflow UI run in Anaconda Prompt:
   cd 'C:\Users\Atharva More\NeuralRetail'
   mlflow ui
   Then open: http://localhost:5000
✅ Day 15-20 MLflow Tracking Complete!
